In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, split, explode, explode_outer
from pyspark.sql import functions as F 

In [3]:
try:
    spark.stop()
except:
    pass  

spark = SparkSession.builder \
    .appName("Session_du_pipeline") \
    .master("yarn") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:8020") \
    .config("spark.hadoop.yarn.resourcemanager.hostname", "resourcemanager") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/08 14:47:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/08 14:47:03 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.
ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/usr/local/lib/python3.10/dist-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/usr/lib/python3.10/socket.py", line 705, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
# Traitement 

In [4]:
try:
    spark.stop()
except:
    pass 
spark = SparkSession.builder\
        .appName('traitement_NLP').getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/08 15:57:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [34]:
spark.version

'3.5.7'

In [35]:
# lecture de nos fichier hdfs 
df = spark.read.option("mergeSchema", "true").parquet("hdfs://namenode:8020/ben/dataLake/")

In [36]:
df.show(5)

+--------------------+----+--------------------+--------------------+----------------+-------------------+--------------------+--------------------+---------------+--------------------+--------------------+
|               poste|lien|          entreprise|              region|contract_propose|date_de_publication|        niveau_etude|   niveau_experience|contrat_propose|          Competence|           formation|
+--------------------+----+--------------------+--------------------+----------------+-------------------+--------------------+--------------------+---------------+--------------------+--------------------+
|Agent Commercial ...|NULL|            CERAGEM |               Dakar|            NULL|         29.07.2025|Bac+1, Bac+2, Bac...|Etudiant, jeune d...|    CDD & Stage|Suivi Commercial ...|<li>La connaissan...|
|  Comptable  - Dakar|NULL|            CERAGEM |               Dakar|            NULL|         29.07.2025|Bac+2, Bac+3, Bac...|Expérience entre ...|    CDD & Stage|Comptabi

In [8]:
df = df.dropDuplicates()

In [9]:
# compter les nombres de lignes qu'on a 
df.count()

413

In [10]:
# voir le schema ou les differents colonnes de nos données 
df.printSchema()

root
 |-- poste: string (nullable = true)
 |-- lien: string (nullable = true)
 |-- entreprise: string (nullable = true)
 |-- region: string (nullable = true)
 |-- contract_propose: string (nullable = true)
 |-- date_de_publication: string (nullable = true)
 |-- niveau_etude: string (nullable = true)
 |-- niveau_experience: string (nullable = true)
 |-- contrat_propose: string (nullable = true)
 |-- Competence: string (nullable = true)
 |-- formation: string (nullable = true)



In [11]:
# Les nombres de differents poste et  entreprises existantes 
df.select("poste").distinct().count()

399

## Colonnes a nettoyer et organiser


In [12]:
# Apres un passage de select, exemple la colonne suivant '"entreprise"
df.select("entreprise").distinct().show(5, truncate=False)

[Stage 14:==============>                                          (3 + 9) / 12]

+----------------------------------------+
|entreprise                              |
+----------------------------------------+
|TECTRA SÉNÉGAL                          |
|MOUABI                                  |
|VISION PLUS VOYAGE                      |
|SOCIÉTÉ DE PROMOTION ET DE CONSTRUCTION |
|NEXGEN TALENT                           |
+----------------------------------------+
only showing top 5 rows



### traitement de chaque colonne

#### formation

In [13]:
# ---------------- pour la colonne formation 

df_formation = df.withColumn(
    "formation_clean",
    F.trim(
        F.regexp_replace(F.col("formation"), r"</li><li>", ", "))
)

df_formation = df_formation.withColumn(
    "formation_clean",
    F.regexp_replace(F.col("formation_clean"), r"</?li>", "")
)

df_formation = df_formation.withColumn(
    "formation_clean",
    F.regexp_replace(F.col("formation_clean"), r"[\n]", "")
)

df_formation = df_formation.withColumn(
    "formation_clean",
    F.regexp_replace(F.col("formation_clean"),r"</?strong>", "")
)

df_formation = df_formation.withColumn(
    "formation_clean",
    F.trim(F.regexp_replace(F.col("formation_clean"), r"<[^>]+>", ""))
)

df_formation.drop("formation")

DataFrame[poste: string, lien: string, entreprise: string, region: string, contract_propose: string, date_de_publication: string, niveau_etude: string, niveau_experience: string, contrat_propose: string, Competence: string, formation_clean: string]

#### niveau etude 

In [14]:
#-------------- Pour la colonne niveau_etude--------------

df_etude = df_formation.withColumn(
    "niveau_etude_clean",
    split(col("niveau_etude"), r"\s*(?:,|&|et)\s*")
)

df_etude = df_etude.drop("niveau_etude")
df_etude = df_etude.drop("formation")

#### contract proposé 

In [15]:
##------------ contract proposé ---------------------------
df_contract = df_etude.withColumn(
    "contract",
    split(col("contrat_propose"), r"\s*(?:,|&|et)\s*")
)

df_contract = df_contract.drop("contrat_propose")

#### region 

In [16]:
### ------------- Pour la region 

df_region = df_contract.withColumn(
    "region_clean",
    F.trim(F.regexp_replace(F.col("region"), r"\s*(?:\n|\t)", ""))

)

#### Experience

In [17]:
# ----- experience 

df_experience = df_region.withColumn(
    "experience", 
    split(col("niveau_experience"), r"\s*(?:,|&|-)\s*")
)

df_experience = df_experience.drop("niveau_experience")

#### Competences 


In [18]:
#---------- Traitement des competences 

df_competence = df_experience.withColumn(
    "competences",
    split(col("Competence"), r"\s*(-)\s*")
)

In [19]:
df_competence.show(3)

[Stage 17:====>                                                   (1 + 11) / 12]

+--------------------+----+----------------+-------------+----------------+-------------------+--------------------+--------------------+--------------------+----------------+-------------+--------------------+--------------------+
|               poste|lien|      entreprise|       region|contract_propose|date_de_publication|          Competence|     formation_clean|  niveau_etude_clean|        contract| region_clean|          experience|         competences|
+--------------------+----+----------------+-------------+----------------+-------------------+--------------------+--------------------+--------------------+----------------+-------------+--------------------+--------------------+
|Ouvrier d’Usine -...|NULL|INTRA INTERIM SN|International|            NULL|         20.06.2025|Accompagnement - ...|Niveau d’anglais ...|[Qualification av...|  [CDI, Intérim]|International|[Etudiant, jeune ...|[Accompagnement, ...|
|Responsable Admin...|NULL|  CABINET CARREE|        Dakar|            NU

## Creation de la premiere dataset qui contient toutes les données 

In [20]:
df_clean = df_competence.drop("Competence")
df_clean.show(5, truncate=False)

+-----------------------------------------------------+----+----------------+-----------------------------------------------------------------+----------------+-------------------+-----------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------+----------------------+-----------------------------------------------------------------+-----------------------------------------+--------------------------------------------------------------------------------------------------------+
|poste                                                |lien|entreprise      |region                                                           |contract_propose|date_de_publication|formation_clean                                                                                                              |niveau_etude_clean                                                     |contrac

In [21]:
df_clean.select('lien').distinct().show()

+--------------------+
|                lien|
+--------------------+
|https://www.emplo...|
|https://www.emplo...|
|https://www.emplo...|
|https://www.emplo...|
|https://www.emplo...|
|https://www.emplo...|
|https://www.emplo...|
|https://www.emplo...|
|https://www.emplo...|
|https://www.emplo...|
|https://www.emplo...|
|https://www.emplo...|
|https://www.emplo...|
|https://www.emplo...|
|https://www.emplo...|
|https://www.emplo...|
|https://www.emplo...|
|https://www.emplo...|
|https://www.emplo...|
|https://www.emplo...|
+--------------------+
only showing top 20 rows



In [22]:
df_clean.select('competences').show(truncate= False)

[Stage 26:==========================================>              (9 + 3) / 12]

+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|competences                                                                                                                                                                                                                                            |
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|[Accompagnement, Assemblage, Emballage, Maintenance, Production, Qualité, Sécurité]                                                                                                                                                                    |


## Creation du deuxieme dataset pour le machine learning

In [24]:
df_clean.select("entreprise","poste","lien").distinct().show(3, truncate=False)

[Stage 29:==============>                                          (3 + 9) / 12]

+------------------+--------------------------------------+--------------------------------------------------------------------------------------------+
|entreprise        |poste                                 |lien                                                                                        |
+------------------+--------------------------------------+--------------------------------------------------------------------------------------------+
|Azalai Hotels     |Responsable des Ventes                |https://www.emploidakar.com/offre-demploi/responsable-des-ventes/                           |
|Plan International|Regional Business Development Director|https://www.emploidakar.com/offre-demploi/regional-business-development-director/           |
|DEXINTEC          |Investigation Officer                 |https://www.emploidakar.com/offre-demploi/dexintec-easy-buy-dakar-cdd-investigation-officer/|
+------------------+--------------------------------------+-----------------------

In [25]:
tab = df_clean.select("entreprise", "poste", 'lien',explode_outer(col("competences")).alias("competence"))

In [26]:
df_filter = tab.filter(col("entreprise").like("%Sama %"))
    #select('poste', 'competence', 'lien').distinct().show()

df_filter.select('poste','competence', 'lien').distinct().show()

[Stage 32:====>                                                   (1 + 11) / 12]

+--------------------+----------+--------------------+
|               poste|competence|                lien|
+--------------------+----------+--------------------+
|03 Coordonnateurs...|      NULL|https://www.emplo...|
|08 Agents Formate...|      NULL|https://www.emplo...|
+--------------------+----------+--------------------+



In [27]:
tab.filter(col("entreprise").like("%Sama%")).show()

+------------------+--------------------+--------------------+----------+
|        entreprise|               poste|                lien|competence|
+------------------+--------------------+--------------------+----------+
|Sama Mbey (myAgro)|03 Coordonnateurs...|https://www.emplo...|      NULL|
|Sama Mbey (myAgro)|08 Agents Formate...|https://www.emplo...|      NULL|
+------------------+--------------------+--------------------+----------+



In [28]:
df_ml = df_clean.select("entreprise", "poste","lien", explode_outer(col("competences")).alias("competence"),
col("formation_clean").alias("formation"), explode_outer(col("niveau_etude_clean")).alias("niveau_etude"),
explode_outer(col("contract")).alias("contrat"), explode_outer(col("experience")).alias("experience"),
col("region_clean").alias("region"), col("date_de_publication").alias("date_de_pulication")
)

In [29]:
df_ml.show(5, truncate=False)

[Stage 41:=========>                                              (2 + 10) / 12]

+----------------+-----------------------------------+----+--------------+-------------------------------------+-----------------------+-------+---------------------+-------------+------------------+
|entreprise      |poste                              |lien|competence    |formation                            |niveau_etude           |contrat|experience           |region       |date_de_pulication|
+----------------+-----------------------------------+----+--------------+-------------------------------------+-----------------------+-------+---------------------+-------------+------------------+
|INTRA INTERIM SN|Ouvrier d’Usine - Roumanie (Europe)|NULL|Accompagnement|Niveau d’anglais intermédiaire requis|Qualification avant bac|CDI    |Etudiant             |International|20.06.2025        |
|INTRA INTERIM SN|Ouvrier d’Usine - Roumanie (Europe)|NULL|Accompagnement|Niveau d’anglais intermédiaire requis|Qualification avant bac|CDI    |jeune diplômé et plus|International|20.06.2025        |


In [30]:
df_ml.select('lien').distinct().show()

+--------------------+
|                lien|
+--------------------+
|https://www.emplo...|
|https://www.emplo...|
|https://www.emplo...|
|https://www.emplo...|
|https://www.emplo...|
|https://www.emplo...|
|https://www.emplo...|
|https://www.emplo...|
|https://www.emplo...|
|https://www.emplo...|
|https://www.emplo...|
|https://www.emplo...|
|https://www.emplo...|
|https://www.emplo...|
|https://www.emplo...|
|https://www.emplo...|
|https://www.emplo...|
|https://www.emplo...|
|https://www.emplo...|
|https://www.emplo...|
+--------------------+
only showing top 20 rows



In [31]:
df_ml.count()

7065

# Nos datasetes

## pour le ml 

In [32]:
print("dataset pour le machine learning et l'analyse analytique")
df_ml.show(10, truncate=False)

dataset pour le machine learning et l'analyse analytique
+----------------+-----------------------------------+----+--------------+-------------------------------------+-----------------------+-------+---------------------+-------------+------------------+
|entreprise      |poste                              |lien|competence    |formation                            |niveau_etude           |contrat|experience           |region       |date_de_pulication|
+----------------+-----------------------------------+----+--------------+-------------------------------------+-----------------------+-------+---------------------+-------------+------------------+
|INTRA INTERIM SN|Ouvrier d’Usine - Roumanie (Europe)|NULL|Accompagnement|Niveau d’anglais intermédiaire requis|Qualification avant bac|CDI    |Etudiant             |International|20.06.2025        |
|INTRA INTERIM SN|Ouvrier d’Usine - Roumanie (Europe)|NULL|Accompagnement|Niveau d’anglais intermédiaire requis|Qualification avant bac|CDI    

## Donnees brutes 

In [33]:
print("dataset normale sur les offres")
df_clean.show(10, truncate=False)

dataset normale sur les offres
+-----------------------------------------------------+----+----------------------+-----------------------------------------------------------------+----------------+-------------------+-----------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------+----------------------------------------------------------------------------+-----------------------------------------------------------------+-----------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|poste                                                |lien|entreprise            |region                                                           |co

# INgection a notre dataWarehouse 

In [41]:
### ---------------------- nos liens --------------------------------------------####
url = "jdbc:postgresql://postgres_warehouse:5432/warehouse_db"
user = "admin"
password = "admin_pwd"
driver = "org.postgresql.Driver"

## Creation de la table du ml 

In [42]:
###---------------Creation du dataset ML --------------------------------####

df_ml.write\
    .format("jdbc")\
    .option("url", url)\
    .option("dbtable","dataset_ml")\
    .option("user", user)\
    .option("password",password)\
    .option("driver", driver)\
    .mode("append")\
    .save()

In [43]:
df_ml.show(truncate=False)

+----------------+-----------------------------------+----+--------------+-------------------------------------+-----------------------+-------+---------------------+-------------+------------------+
|entreprise      |poste                              |lien|competence    |formation                            |niveau_etude           |contrat|experience           |region       |date_de_pulication|
+----------------+-----------------------------------+----+--------------+-------------------------------------+-----------------------+-------+---------------------+-------------+------------------+
|INTRA INTERIM SN|Ouvrier d’Usine - Roumanie (Europe)|NULL|Accompagnement|Niveau d’anglais intermédiaire requis|Qualification avant bac|CDI    |Etudiant             |International|20.06.2025        |
|INTRA INTERIM SN|Ouvrier d’Usine - Roumanie (Europe)|NULL|Accompagnement|Niveau d’anglais intermédiaire requis|Qualification avant bac|CDI    |jeune diplômé et plus|International|20.06.2025        |


### Creation de la table simple (normmale) 

In [44]:
##########___-------Mise en forme----------__________________________###
df_propres = df_clean.select("entreprise", "poste", col("competences").alias("competence"),
col("formation_clean").alias("formation"), col("niveau_etude_clean").alias("niveau_etude"),
col("contract").alias("contrat"), col("experience").alias("experience"),"lien",
col("region_clean").alias("region"), col("date_de_publication").alias("date_de_publication")
)

In [45]:
df_propres.write\
    .format("jdbc")\
    .option("url", url)\
    .option("dbtable","offres_emploi")\
    .option("user", user)\
    .option("password",password)\
    .option("driver", driver)\
    .mode("append")\
    .save()


In [46]:
df_lecture = spark.read \
    .format("jdbc") \
    .option("url", url) \
    .option("dbtable", "offres_emploi") \
    .option("user", user) \
    .option("password", password) \
    .option("driver", driver) \
    .load()

In [47]:
df_lecture.show(5, truncate=False)

+----------------+-----------------------------------------------------+--------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------+----------------------+-----------------------------------------+----+-----------------------------------------------------------------+-------------------+
|entreprise      |poste                                                |competence                                                                                              |formation                                                                                                                    |niveau_etude                                                           |contrat               |experience                               |lien|region                 

In [48]:
df_result = df_lecture.select("date_de_publication")

df_result.distinct().show(100, truncate=False)


+----------------------+
|date_de_publication   |
+----------------------+
|Publié le 23 mars 2026|
|16.06.2025            |
|30.06.2025            |
|29.07.2025            |
|04.07.2025            |
|08.08.2025            |
|02.08.2025            |
|11.06.2025            |
|12.06.2025            |
|25.07.2025            |
|Publié le 30 mars 2026|
|05.08.2025            |
|16.07.2025            |
|06.08.2025            |
|13.06.2025            |
|11.07.2025            |
|Publié le 7 avril 2026|
|20.06.2025            |
|21.07.2025            |
|01.07.2025            |
|Publié le 1 avril 2026|
|Publié le 8 avril 2026|
|10.06.2025            |
|10.07.2025            |
|Publié le 25 mars 2026|
|02.07.2025            |
|Publié le 6 avril 2026|
|25.06.2025            |
|07.07.2025            |
|21.06.2025            |
|07.08.2025            |
|Publié le 27 mars 2026|
|31.07.2025            |
|19.07.2025            |
|15.07.2025            |
|04.08.2025            |
|17.06.2025            |
